In [ ]:
import torch
torch.__version__
import os
os.environ['TRITON_INTERPRET'] = '1' # needs to be set *before* triton is imported

In [ ]:
torch.cuda.is_available()

In [ ]:
import triton
import triton.language as tl

In [ ]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [ ]:
DEVICE

In [ ]:
@triton.jit
def gray_scale_kernel(x_ptr, out_ptr, h, w, bs0: tl.constexpr, bs1: tl.constexpr):
    pid_w = tl.program_id(0)
    pid_h = tl.program_id(1)

    # 1-d vector offsets
    offs_w = pid_w * bs0 + tl.arange(0, bs0)
    offs_h = pid_h * bs1 + tl.arange(0, bs1)

    # 2-d matrix offsets
    offs = w * offs_w[:, None] + offs_h[None, :]

    # 2-d mask, doesn't must go beyond either axis, therefore 'logical and'
    mask = (offs_w < w)[:, None] & (offs_h < h)[None, :]

    r = tl.load(x_ptr + 0 * w * h * offs, mask = mask) # r is in channel-0
    g = tl.load(x_ptr + 1 * w * h * offs, mask = mask)# g is in channel-1
    b = tl.load(x_ptr + 2 * w * h * offs, mask = mask)# b is in channel-2

    out = 0.2989*r + 0.5870*g + 0.1140*b

    # write to HBM
    tl.store(out_ptr + offs, out, mask = mask)

In [ ]:
# Helper function
def gray_scale(x, bs):
    c, h, w = x.shape
    gray = torch.empty_like(x, dtype=x.dtype, device=DEVICE)
    # grid is 2-D here
    # having a grid function is useful when we benchmark and auto-tune kernels
    grid = lambda meta: (tl.cdiv(w, meta['bs0']), tl.cdiv(h, meta['bs1']))
    gray_scale_kernel[grid](x, gray, h, w, bs0=bs[0], bs1=bs[1])
    return gray.view(h,w)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
path_img = Path('/home/mani/cuda/triton/cute-dog.jpg')

In [ ]:
import torchvision as tv
import torchvision.transforms.functional as tvf
from torchvision import io

In [ ]:
img = io.decode_image(path_img)

In [ ]:
img.shape

In [ ]:
img.stride()

In [ ]:
gray_img = gray_scale(img.to(DEVICE), bs=(32,32))

In [ ]:
rimg = img[0]

In [ ]:
rimg.shape

In [ ]:
rimg = rimg[None,:]

In [ ]:
rimg.shape

In [ ]:
raw_bytes = io.encode_jpeg(rimg)

In [ ]:
if isinstance(raw_bytes, torch.Tensor):
    raw_bytes = raw_bytes.cpu().numpy().tobytes()

In [ ]:
with open('raw_image.jpeg', 'wb') as file:
    file.write(raw_bytes)

In [ ]:
#

In [ ]:
x  = torch.rand((4,3))

In [ ]:
x.shape

In [ ]:
#

In [ ]:
#

In [ ]:
#

In [ ]:
#